### Review LangChain Basics

In [59]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.schema.runnable import RunnablePassthrough, RunnableLambda, RunnableParallel
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

from dotenv import load_dotenv
from typing import List
from pydantic import BaseModel, Field
import os

In [30]:
load_dotenv()
llm = ChatOpenAI(
    model_name="gpt-4o-mini",
    temperature=0
)

In [14]:
llm_response = llm.invoke("Tell me a Joke!")
print(llm_response)

content='Why did the scarecrow win an award?\n\nBecause he was outstanding in his field!' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 12, 'total_tokens': 29, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_560af6e559', 'id': 'chatcmpl-CXwQaHTOmkg1W45xnLaL3ZxTM86MM', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='run--8c71c669-2218-4c1e-a848-08fc7d928fb4-0' usage_metadata={'input_tokens': 12, 'output_tokens': 17, 'total_tokens': 29, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


### Parsing output

In [15]:
parser = StrOutputParser()
parser.invoke(llm_response)

'Why did the scarecrow win an award?\n\nBecause he was outstanding in his field!'

### Simple chain

In [16]:
simple_chain = llm | parser
response = simple_chain.invoke("Tell me a Joke!")
print(response)

Why did the scarecrow win an award?

Because he was outstanding in his field!


### Structured Output

In [17]:
class MobileReview(BaseModel):
    phone_model: str = Field(description="Name and model of the phone")
    rating: float = Field(description="Rating from 1 to 5")
    pros: List[str] = Field(description="Positive aspects of the phone")
    cons: List[str] = Field(description="Negative aspects of the phone")
    review_summary: str = Field(description="Summary of the review")

review_text = """
Just got my hands on google pixel fold, it's a beast! the screen is gorgeous, colors pops like crazy.
I've been using it for a few days now and I'm loving it. I can't wait to get home and show it off to my family.
The camera quality is outstanding, even in low light conditions. It captures everything perfectly.
But I must say that the battery life is a bit of a concern. It's not as long as I expected, but that's not a big deal.
Overall, I'm really impressed with this phone it is solid 4 star rating. It's a great value for the money and I highly recommend it.
"""

structured_llm = llm.with_structured_output(MobileReview)

structured_response = structured_llm.invoke(review_text)
print(structured_response)

phone_model='Google Pixel Fold' rating=4.0 pros=['Gorgeous screen with vibrant colors', 'Outstanding camera quality, even in low light', 'Great value for the money'] cons=['Battery life could be better'] review_summary="Overall, I'm really impressed with the Google Pixel Fold. It's a solid device with a stunning display and excellent camera performance, though the battery life is a bit of a concern."


### Prompt Template

- Helps create dynamic prompts

In [ ]:
prompt = ChatPromptTemplate.from_template(
    "Tell me a short story about {topic}"
)

prompt.invoke("Space")

ChatPromptValue(messages=[HumanMessage(content='Tell me a short story about Space', additional_kwargs={}, response_metadata={})])

In [8]:
chain_with_prompt = prompt | llm | parser
response = chain_with_prompt.invoke("Cars")
print(response)

Once upon a time in the bustling town of Autoville, cars weren’t just machines; they were vibrant characters with personalities. Each car had its own quirks and dreams. Among them was a little blue hatchback named Benny. Benny was small and often overlooked, but he had a big heart and an even bigger dream: to race in the annual Autoville Grand Prix.

Every year, the Grand Prix attracted the fastest and flashiest cars, like the sleek red sports car, Blaze, and the powerful black muscle car, Titan. Benny admired them from afar, wishing he could join the race. His friends, a wise old van named Vinnie and a cheerful electric car named Zoe, encouraged him to believe in himself.

“Benny, it’s not about size or speed; it’s about heart and determination,” Vinnie said, his headlights twinkling with wisdom.

With newfound confidence, Benny decided to enter the race. He spent weeks preparing, tuning his engine and practicing on the winding roads of Autoville. The day of the Grand Prix arrived, an

### Different LLM Messages

In [ ]:
system_message = SystemMessage(
    content="You are a helpful assistant that translates English to French."
)
human_message = HumanMessage(content="I love programming.")

load_dotenv()
parser = StrOutputParser()
llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0)

response = llm.invoke([system_message, human_message])
print(response)

content="J'aime la programmation." additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 5, 'prompt_tokens': 26, 'total_tokens': 31, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_560af6e559', 'id': 'chatcmpl-CXgd4wL2TALWC7ifKT9nUi7RmPtjm', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='run--66440432-6b23-4ba0-b990-ccf29c311a39-0' usage_metadata={'input_tokens': 26, 'output_tokens': 5, 'total_tokens': 31, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


In [11]:
prompt = ChatPromptTemplate(
    [
        ("system", "You are an helpful assistant that tell amazing jokes."),
        ("human", "Tell me a joke about {topic}."),
    ]
)

prompt_value = prompt.invoke({"topic": "Space"})
print(prompt_value)

response = llm.invoke(prompt_value)
print(response)

messages=[SystemMessage(content='You are an helpful assistant that tell amazing jokes.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Tell me a joke about Space.', additional_kwargs={}, response_metadata={})]
content='Why did the sun go to school?\n\nTo get a little brighter!' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 28, 'total_tokens': 42, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_560af6e559', 'id': 'chatcmpl-CXgjJVGf9hff8s4uQpvz5lO6kiZqB', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='run--34125f19-1eb8-4a3b-9007-566bde5676fd-0' usage_metadata={'input_tokens': 28, 'output_tokens': 14, 'total_tokens': 42, 'input_token_details': {'audio': 0

### RAG

In [18]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,
    length_function=len,
)

docx_loader = Docx2txtLoader("/Users/srinivas/Documents/Others/My_projects/Python/DSAIML/GenAI/15_LC_RAG_cover_chat/docs/GreenGrow Innovations_ Company History.docx")
document = docx_loader.load()

splits = text_splitter.split_documents(document)

print(f"Number of documents: {len(splits)}")

Number of documents: 2


In [19]:
splits[0]

Document(metadata={'source': '/Users/srinivas/Documents/Others/My_projects/Python/DSAIML/GenAI/15_LC_RAG_cover_chat/docs/GreenGrow Innovations_ Company History.docx'}, page_content='GreenGrow Innovations was founded in 2010 by Sarah Chen and Michael Rodriguez, two agricultural engineers with a passion for sustainable farming. The company started in a small garage in Portland, Oregon, with a simple mission: to make farming more environmentally friendly and efficient.\n\n\n\nIn its early days, GreenGrow focused on developing smart irrigation systems that could significantly reduce water usage in agriculture. Their first product, the WaterWise Sensor, was launched in 2012 and quickly gained popularity among local farmers. This success allowed the company to expand its research and development efforts.\n\n\n\nBy 2015, GreenGrow had outgrown its garage origins and moved into a proper office and research facility in the outskirts of Portland. This move coincided with the development of their

In [20]:
splits[0].page_content

'GreenGrow Innovations was founded in 2010 by Sarah Chen and Michael Rodriguez, two agricultural engineers with a passion for sustainable farming. The company started in a small garage in Portland, Oregon, with a simple mission: to make farming more environmentally friendly and efficient.\n\n\n\nIn its early days, GreenGrow focused on developing smart irrigation systems that could significantly reduce water usage in agriculture. Their first product, the WaterWise Sensor, was launched in 2012 and quickly gained popularity among local farmers. This success allowed the company to expand its research and development efforts.\n\n\n\nBy 2015, GreenGrow had outgrown its garage origins and moved into a proper office and research facility in the outskirts of Portland. This move coincided with the development of their second major product, the SoilHealth Monitor, which used advanced sensors to analyze soil composition and provide real-time recommendations for optimal crop growth.'

In [21]:
splits[0].metadata

{'source': '/Users/srinivas/Documents/Others/My_projects/Python/DSAIML/GenAI/15_LC_RAG_cover_chat/docs/GreenGrow Innovations_ Company History.docx'}

#### Let us now load all the documents present based on their type

In [22]:
def load_multiple_documents(directory_path: str) -> List[Document]:

    documents = []

    for file in os.listdir(directory_path):
        if file.endswith(".docx"):
            loader = Docx2txtLoader(os.path.join(directory_path, file))
        elif file.endswith(".pdf"):
            loader = PyPDFLoader(os.path.join(directory_path, file))
        else:
            print(f"Skipping {file} as it is not a supported file type")
            continue
        documents.extend(loader.load())

    return documents

document_path = "docs"
documents = load_multiple_documents(document_path)

print(f"Number of documents: {len(documents)}")
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100, length_function=len)
splits = text_splitter.split_documents(documents)

print(f"Number of documents after splitting: {len(splits)}")


Number of documents: 5
Number of documents after splitting: 8


In [23]:
embeddings = OpenAIEmbeddings()
doc_embeddings = embeddings.embed_documents([split.page_content for split in splits])
print(f"Embeddings created for {len(doc_embeddings)} documents")

Embeddings created for 8 documents


In [24]:
len(doc_embeddings[0])

1536

### store the chunks/splits in chroma db

In [ ]:
collection_name = "docs_collection"

vectorstore = Chroma.from_documents(splits, embeddings, collection_name=collection_name)

print("Document embeddings added to ChromaDB present in ./chroma_db directory")

Document embeddings added to ChromaDB present in ./chromadb directory


Perform similarity search

In [ ]:
query = "What is GreenGrow Innovations?"
search_results = vectorstore.similarity_search(query, k = 2)

for index, result in enumerate(search_results):
    print(f"Result {index}: {result.page_content}")

Result 0: The company's breakthrough came in 2018 with the introduction of the EcoHarvest System, an integrated solution that combined smart irrigation, soil monitoring, and automated harvesting techniques. This system caught the attention of large-scale farmers across the United States, propelling GreenGrow to national prominence.



Today, GreenGrow Innovations employs over 200 people and has expanded its operations to include offices in California and Iowa. The company continues to focus on developing sustainable agricultural technologies, with ongoing projects in vertical farming, drought-resistant crop development, and AI-powered farm management systems.



Despite its growth, GreenGrow remains committed to its original mission of promoting sustainable farming practices. The company regularly partners with universities and research institutions to advance the field of agricultural technology and hosts annual conferences to share knowledge with farmers and other industry profession

In [35]:
query = "When was GreenGrow Innovations founded?"
search_results = vectorstore.similarity_search(query, k = 2)

for index, result in enumerate(search_results):
    print(f"Result {index}: {result.page_content}")

Result 0: The company's breakthrough came in 2018 with the introduction of the EcoHarvest System, an integrated solution that combined smart irrigation, soil monitoring, and automated harvesting techniques. This system caught the attention of large-scale farmers across the United States, propelling GreenGrow to national prominence.



Today, GreenGrow Innovations employs over 200 people and has expanded its operations to include offices in California and Iowa. The company continues to focus on developing sustainable agricultural technologies, with ongoing projects in vertical farming, drought-resistant crop development, and AI-powered farm management systems.



Despite its growth, GreenGrow remains committed to its original mission of promoting sustainable farming practices. The company regularly partners with universities and research institutions to advance the field of agricultural technology and hosts annual conferences to share knowledge with farmers and other industry profession

In [39]:
# But we cannot use similarity_search directly as it does not support invoke method directly.
# So, we need to use retriever to do the same.

retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
retrieved_docs = retriever.invoke(query)

print(retrieved_docs)


[Document(id='6dd9a866-edde-41c1-a6dc-f50e29ee3b07', metadata={'source': 'docs/GreenGrow Innovations_ Company History.docx'}, page_content="The company's breakthrough came in 2018 with the introduction of the EcoHarvest System, an integrated solution that combined smart irrigation, soil monitoring, and automated harvesting techniques. This system caught the attention of large-scale farmers across the United States, propelling GreenGrow to national prominence.\n\n\n\nToday, GreenGrow Innovations employs over 200 people and has expanded its operations to include offices in California and Iowa. The company continues to focus on developing sustainable agricultural technologies, with ongoing projects in vertical farming, drought-resistant crop development, and AI-powered farm management systems.\n\n\n\nDespite its growth, GreenGrow remains committed to its original mission of promoting sustainable farming practices. The company regularly partners with universities and research institutions 

In [41]:
for index, result in enumerate(retrieved_docs):
    print(f"======= Result {index+1} =======")
    print(f"Source document path: {result.metadata["source"]}")
    print(f"Content: {result.page_content}")
    print("\n")

======= Result 1 =======
Source document path: docs/GreenGrow Innovations_ Company History.docx
Content: The company's breakthrough came in 2018 with the introduction of the EcoHarvest System, an integrated solution that combined smart irrigation, soil monitoring, and automated harvesting techniques. This system caught the attention of large-scale farmers across the United States, propelling GreenGrow to national prominence.



Today, GreenGrow Innovations employs over 200 people and has expanded its operations to include offices in California and Iowa. The company continues to focus on developing sustainable agricultural technologies, with ongoing projects in vertical farming, drought-resistant crop development, and AI-powered farm management systems.



Despite its growth, GreenGrow remains committed to its original mission of promoting sustainable farming practices. The company regularly partners with universities and research institutions to advance the field of agricultural techno

### LLM Call 

Here in the above part of the code we were able to retieve the documents, but we still have to get the answer, for this we need to make a call to LLM

In [38]:
template = """
Answer the question as truthfully as possible using the provided context, 
and if the answer is not contained within the context, say "I don't know"

Context: {context}
Question: {question}

Answer:
"""

prompt = ChatPromptTemplate.from_template(template)

In [44]:
context_text = "\n\n".join([doc.page_content for doc in retrieved_docs])
question = "When was GreenGrow Innovations founded?"

In [45]:
final_prompt = prompt.invoke({"context": context_text, "question":question})

In [46]:
print(final_prompt)

messages=[HumanMessage(content='\nAnswer the question as truthfully as possible using the provided context, \nand if the answer is not contained within the context, say "I don\'t know"\n\nContext: The company\'s breakthrough came in 2018 with the introduction of the EcoHarvest System, an integrated solution that combined smart irrigation, soil monitoring, and automated harvesting techniques. This system caught the attention of large-scale farmers across the United States, propelling GreenGrow to national prominence.\n\n\n\nToday, GreenGrow Innovations employs over 200 people and has expanded its operations to include offices in California and Iowa. The company continues to focus on developing sustainable agricultural technologies, with ongoing projects in vertical farming, drought-resistant crop development, and AI-powered farm management systems.\n\n\n\nDespite its growth, GreenGrow remains committed to its original mission of promoting sustainable farming practices. The company regul

In [47]:
answer = llm.invoke(final_prompt)
print(answer)

content='GreenGrow Innovations was founded in 2010.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 382, 'total_tokens': 392, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_560af6e559', 'id': 'chatcmpl-CXx33xBtKf9SRs5tn8HZa8k38GW4M', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='run--7925dff0-6fa9-4337-a4ed-11d9514636f5-0' usage_metadata={'input_tokens': 382, 'output_tokens': 10, 'total_tokens': 392, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


### This is using runnable passthrough to create a single chain

In [53]:
def docs_to_context(documents: List[Document]) -> str:
    return "\n\n".join([doc.page_content for doc in documents])

In [ ]:
rag_chain = (
    {"context": retriever | RunnableLambda(docs_to_context), "question": RunnablePassthrough()} | prompt | llm)

rag_chain.invoke("When was GreenGrow Innovations founded?")

AIMessage(content='GreenGrow Innovations was founded in 2010.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 382, 'total_tokens': 392, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_560af6e559', 'id': 'chatcmpl-CXxIur6m2bpRwvmtJGvuuzq9OATbx', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--b418bf02-32c2-4c15-b962-9554220af804-0', usage_metadata={'input_tokens': 382, 'output_tokens': 10, 'total_tokens': 392, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

Another cleaner way to do this


In [56]:
rag_parallel_chain = RunnableParallel({
    "context": retriever | RunnableLambda(docs_to_context),
    "question": RunnablePassthrough()
})

In [57]:
parser = StrOutputParser()

In [58]:
main_chain = rag_parallel_chain | prompt | llm | parser
main_chain.invoke("When was GreenGrow Innovations founded?")

'GreenGrow Innovations was founded in 2010.'

### Conclusion

Here we built a Question and Answering RAG bot